# DPO: Direct Preference Optimization

> Implement DPO for aligning LLMs without explicit reward model

**DPO Paper:** Rafailov et al., 2023 — *Direct Preference Optimization: Your Language Model is Secretly a Reward Model*

**Key Insight:** Instead of training a separate reward model (like RLHF), DPO directly optimizes the policy using preference data.

**Loss Function:**
```
L_DPO = -log σ(β * [log(π_θ(y_w|x) / π_ref(y_w|x)) - log(π_θ(y_l|x) / π_ref(y_l|x))])
```
Where:
- `y_w` = preferred (winning) response
- `y_l` = rejected (losing) response
- `π_θ` = current policy (being trained)
- `π_ref` = reference policy (frozen SFT model)
- `β` = temperature parameter controlling divergence from reference

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict
from dataclasses import dataclass

## 1. Preference Data Structure

In [ ]:
@dataclass
class PreferencePair:
    """A preference pair: prompt + chosen + rejected"""
    prompt: str
    chosen: str   # Preferred response (y_w)
    rejected: str # Less preferred response (y_l)

# Sample preference dataset
preference_data = [
    PreferencePair(
        prompt="Explain photosynthesis.",
        chosen="Photosynthesis is the process by which plants convert sunlight, water, and carbon dioxide into glucose and oxygen. Chlorophyll in leaves captures light energy to drive this chemical reaction.",
        rejected="Plants make food from sun. They need water and air."
    ),
    PreferencePair(
        prompt="How do I bake sourdough bread?",
        chosen="1. Feed your starter 4-12 hours before baking. 2. Mix flour, water, starter, and salt. 3. Stretch and fold every 30 min for 2 hours. 4. Bulk ferment 4-6 hours. 5. Shape and proof overnight in fridge. 6. Bake at 450°F for 45 min with steam.",
        rejected="Just mix flour and water and bake it."
    ),
    PreferencePair(
        prompt="What causes climate change?",
        chosen="Climate change is primarily driven by greenhouse gas emissions from burning fossil fuels, deforestation, and industrial processes. These gases trap heat in Earth's atmosphere, causing global temperatures to rise.",
        rejected="The weather just changes naturally sometimes."
    ),
    PreferencePair(
        prompt="Write a Python function to check if a number is prime.",
        chosen="```python\ndef is_prime(n):\n    if n < 2: return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0: return False\n    return True\n```",
        rejected="Use a library function."
    ),
    PreferencePair(
        prompt="Why is the sky blue?",
        chosen="The sky appears blue due to Rayleigh scattering. Shorter blue wavelengths of sunlight scatter more in Earth's atmosphere than longer red wavelengths.",
        rejected="Because it reflects the ocean."
    ),
]

print(f"Preference dataset: {len(preference_data)} pairs")
for i, pair in enumerate(preference_data[:2]):
    print(f"\n--- Pair {i+1} ---")
    print(f"Prompt: {pair.prompt}")
    print(f"Chosen: {pair.chosen[:60]}...")
    print(f"Rejected: {pair.rejected[:60]}...")

## 2. Simplified Language Model (Policy)

In [ ]:
class SimplePolicy:
    """
    Simplified policy model for DPO demonstration.
    In practice, this would be a full transformer.
    """
    
    def __init__(self, vocab_size: int = 100, embed_dim: int = 64):
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        
        # Embedding layer
        self.token_emb = np.random.randn(vocab_size, embed_dim) * 0.02
        
        # Output projection
        self.W_out = np.random.randn(embed_dim, vocab_size) * 0.02
        self.b_out = np.zeros(vocab_size)
        
        # Simple attention-like weights
        self.W_attn = np.random.randn(embed_dim, embed_dim) * 0.02
    
    def encode(self, text: str) -> np.ndarray:
        """Simple character encoding"""
        tokens = [ord(c) % self.vocab_size for c in text]
        return np.array(tokens)
    
    def forward(self, tokens: np.ndarray) -> np.ndarray:
        """Forward pass: return logits"""
        # Embedding
        h = self.token_emb[tokens]  # (seq_len, embed_dim)
        
        # Simple self-attention-like aggregation
        h = h @ self.W_attn
        h = np.maximum(0, h)  # ReLU
        
        # Mean pooling
        h_pooled = np.mean(h, axis=0)
        
        # Output logits
        logits = h_pooled @ self.W_out + self.b_out
        
        return logits
    
    def log_prob(self, prompt: str, response: str) -> float:
        """
        Compute log probability of response given prompt.
        Simplified: use pooled representation of full text.
        """
        full_text = prompt + " " + response
        tokens = self.encode(full_text)
        logits = self.forward(tokens)
        
        # Simplified log-prob: use response tokens
        response_tokens = self.encode(response)
        log_probs = []
        
        for token in response_tokens:
            # Get logit for this token
            token_logit = logits[token % self.vocab_size]
            # Simplified log-prob (not real softmax, just for demo)
            log_probs.append(token_logit)
        
        return np.mean(log_probs)
    
    def copy(self):
        """Create a copy of the policy"""
        new_policy = SimplePolicy(self.vocab_size, self.embed_dim)
        new_policy.token_emb = self.token_emb.copy()
        new_policy.W_out = self.W_out.copy()
        new_policy.b_out = self.b_out.copy()
        new_policy.W_attn = self.W_attn.copy()
        return new_policy
    
    def update(self, grads: Dict, lr: float = 0.01):
        """Update parameters with gradients"""
        self.token_emb -= lr * grads.get('token_emb', 0)
        self.W_out -= lr * grads.get('W_out', 0)
        self.b_out -= lr * grads.get('b_out', 0)
        self.W_attn -= lr * grads.get('W_attn', 0)

# Initialize policy and reference
policy = SimplePolicy(vocab_size=128, embed_dim=64)
reference = policy.copy()  # Frozen reference

print("Policy initialized")
print(f"Parameters: token_emb={policy.token_emb.shape}, W_out={policy.W_out.shape}")

## 3. DPO Loss Function

In [ ]:
def dpo_loss(
    policy: SimplePolicy,
    reference: SimplePolicy,
    pair: PreferencePair,
    beta: float = 0.1
) -> Tuple[float, Dict]:
    """
    Compute DPO loss for a single preference pair.
    
    L_DPO = -log σ(β * [log(π_θ(y_w)/π_ref(y_w)) - log(π_θ(y_l)/π_ref(y_l))])
    """
    # Compute log probabilities
    pi_chosen = policy.log_prob(pair.prompt, pair.chosen)
    pi_rejected = policy.log_prob(pair.prompt, pair.rejected)
    
    ref_chosen = reference.log_prob(pair.prompt, pair.chosen)
    ref_rejected = reference.log_prob(pair.prompt, pair.rejected)
    
    # Log ratios
    ratio_chosen = pi_chosen - ref_chosen
    ratio_rejected = pi_rejected - ref_rejected
    
    # DPO objective
    logits = beta * (ratio_chosen - ratio_rejected)
    
    # Sigmoid loss: -log σ(logits)
    # For numerical stability: -log σ(x) = log(1 + exp(-x))
    if logits >= 0:
        loss = np.log(1 + np.exp(-logits))
    else:
        loss = -logits + np.log(1 + np.exp(logits))
    
    info = {
        'pi_chosen': pi_chosen,
        'pi_rejected': pi_rejected,
        'ref_chosen': ref_chosen,
        'ref_rejected': ref_rejected,
        'ratio_chosen': ratio_chosen,
        'ratio_rejected': ratio_rejected,
        'logits': logits,
        'loss': loss,
    }
    
    return loss, info

# Test DPO loss
test_pair = preference_data[0]
loss, info = dpo_loss(policy, reference, test_pair, beta=0.1)

print(f"DPO Loss: {loss:.4f}")
print(f"\nComponents:")
for k, v in info.items():
    print(f"  {k}: {v:.4f}")

## 4. DPO Training Loop

In [ ]:
def compute_gradients(policy: SimplePolicy, pair: PreferencePair, beta: float = 0.1) -> Dict:
    """
    Compute approximate gradients for DPO.
    In practice, use autograd (PyTorch/TensorFlow).
    """
    eps = 1e-4
    
    # Compute loss at current point
    loss_0, _ = dpo_loss(policy, reference, pair, beta)
    
    grads = {}
    
    # Approximate gradient for W_out (simplified)
    policy.W_out += eps
    loss_plus, _ = dpo_loss(policy, reference, pair, beta)
    policy.W_out -= eps
    
    grads['W_out'] = (loss_plus - loss_0) / eps
    
    return grads

# Training configuration
beta = 0.1
lr = 0.001
epochs = 100

losses = []
chosen_probs = []
rejected_probs = []

print("Starting DPO training...")

for epoch in range(epochs):
    epoch_loss = 0
    
    for pair in preference_data:
        loss, info = dpo_loss(policy, reference, pair, beta)
        epoch_loss += loss
        
        # Simple parameter update (simplified)
        # In practice: use proper backprop
        if epoch < epochs - 1:
            # Small random perturbation to simulate learning
            policy.W_out += lr * (info['ratio_chosen'] - info['ratio_rejected']) * 0.01
    
    avg_loss = epoch_loss / len(preference_data)
    losses.append(avg_loss)
    
    # Track probabilities
    test_pair = preference_data[0]
    cp = policy.log_prob(test_pair.prompt, test_pair.chosen)
    rp = policy.log_prob(test_pair.prompt, test_pair.rejected)
    chosen_probs.append(cp)
    rejected_probs.append(rp)
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | Loss: {avg_loss:.4f} | Chosen: {cp:.3f} | Rejected: {rp:.3f}")

print(f"\nFinal - Chosen prob: {chosen_probs[-1]:.3f}, Rejected prob: {rejected_probs[-1]:.3f}")

## 5. Training Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss curve
axes[0].plot(losses, color='#FF6B6B', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('DPO Loss')
axes[0].set_title('DPO Training Loss')
axes[0].grid(alpha=0.3)

# Log probability comparison
axes[1].plot(chosen_probs, label='Chosen (y_w)', color='#4ECDC4', linewidth=2)
axes[1].plot(rejected_probs, label='Rejected (y_l)', color='#FF6B6B', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Log Probability')
axes[1].set_title('Policy Log Probabilities')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Margin (chosen - rejected)
margin = np.array(chosen_probs) - np.array(rejected_probs)
axes[2].plot(margin, color='#45B7D1', linewidth=2)
axes[2].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Log Prob Margin')
axes[2].set_title('Preference Margin (y_w - y_l)')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. DPO vs RLHF Comparison

In [ ]:
print("📊 DPO vs RLHF Comparison:")
print("=" * 60)

comparison = {
    'Aspect': ['Reward Model', 'Training Steps', 'Memory', 'Stability', 'Hyperparameters', 'Implementation'],
    'RLHF': [
        'Separate model needed',
        '3 stages (SFT → RM → PPO)',
        'High (multiple models)',
        'Unstable (PPO)',
        'Many (KL, clip, etc.)',
        'Complex'
    ],
    'DPO': [
        'Implicit (no separate model)',
        '1 stage (direct)',
        'Low (policy + reference)',
        'Stable (classification)',
        'Few (β only)',
        'Simple'
    ]
}

for i in range(len(comparison['Aspect'])):
    print(f"\n{comparison['Aspect'][i]}:")
    print(f"  RLHF: {comparison['RLHF'][i]}")
    print(f"  DPO:  {comparison['DPO'][i]}")

# Beta parameter effect
betas = [0.01, 0.05, 0.1, 0.5, 1.0]
beta_losses = []

for b in betas:
    test_policy = reference.copy()
    loss, _ = dpo_loss(test_policy, reference, preference_data[0], beta=b)
    beta_losses.append(loss)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(betas, beta_losses, marker='o', color='#45B7D1', linewidth=2, markersize=8)
ax.set_xlabel('β (Beta)')
ax.set_ylabel('Initial Loss')
ax.set_title('Effect of Beta Parameter on DPO Loss')
ax.set_xscale('log')
ax.grid(alpha=0.3)

for b, l in zip(betas, beta_losses):
    ax.annotate(f'β={b}', (b, l), textcoords="offset points", xytext=(0, 10), ha='center')

plt.tight_layout()
plt.show()

## 🎯 Exercises

1. **Beta Tuning**: Experiment with β values. Higher β = more divergence from reference.
2. **IPO**: Implement Identity Preference Optimization (IPO), a variant of DPO.
3. **KTO**: Implement Kahneman-Tversky Optimization (KTO), another DPO variant.
4. **Real Model**: Apply DPO to a real model using `trl` library.
5. **Pair Quality**: Analyze how preference pair quality affects DPO performance.
6. **Length Bias**: Investigate and mitigate length bias in DPO.